# Same-Hessian order and mechanism ablation (review of the ESANN 2027 draft)
Runtime → Change runtime type → **T4 GPU**, then Run all. Results are appended to Google Drive after every layer, so re-running after a disconnect resumes where it stopped.

For every linear layer of every 4th decoder block, the Hessian is computed once from the dense model on the paper's calibration windows, and the layer is compressed with the paper's three orders and four ablation arms using that one Hessian (definitions in `arms.py`). Estimated time on a T4: about 20 min for SmolLM2-135M and 40 min each for SmolLM2-360M and Qwen2.5-0.5B.


In [ ]:
# 1. environment and the paper's code
!pip install -q nvidia-ml-py
!git clone -q https://github.com/antounyaacoub/esann2027-compose || true
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')


In [ ]:
%%writefile arms.py
"""Compression arms for the order and mechanism ablation, built only from the paper's obs_sweep.

Every arm starts from the same dense weights W and the same Hessian H. Arms differ only in the order
of the two operations, in which mask is used, and in which quantization grid is used:

    arm                    first sweep          second sweep                   mask chosen on   grid fitted on
    prune      (E_P)       prune W              -                              W                -
    quant      (E_Q)       quantize W           -                              -                W
    joint                  prune + quantize W   -                              W (per block)    kept weights
    p_then_q               prune W -> W_P       quantize W_P, mask M_P fixed   W                kept weights
    q_then_p               quantize W -> W_Q    prune W_Q, grid G_Q fixed      W_Q              W (stale)
    q_then_p_refit         quantize W -> W_Q    prune W_Q, grid re-fitted      W_Q              kept weights
    q_then_p_pmask         quantize W -> W_Q    mask M_P, grid G_Q fixed       W (M_P)          W (stale)
    q_then_p_pmask_refit   quantize W -> W_Q    mask M_P, grid re-fitted       W (M_P)          kept weights
    p_then_q_stale         prune W -> W_P       quantize W_P, grid G_Q fixed   W                W (stale)

joint, p_then_q and q_then_p are exactly the paper's compress_weight orders (check_against_core
asserts it). Reading the ablation: if q_then_p_refit closes most of the gap between q_then_p and
p_then_q, the stale grid explains the quantize-first penalty (the paper's reading, Sec. 3); if
q_then_p_pmask[_refit] closes it, mask selection on rounded weights does (Harma et al.'s mechanism).
p_then_q_stale is the converse check: a stale grid imposed on the prune-first order.
"""
import types
import torch

PAPER_ARMS = ("joint", "p_then_q", "q_then_p")
ABLATION_ARMS = ("q_then_p_refit", "q_then_p_pmask", "q_then_p_pmask_refit", "p_then_q_stale")
ALL_ARMS = PAPER_ARMS + ABLATION_ARMS


def as_core(obj):
    """Accept the paper's core module, or any namespace (e.g. notebook globals) holding obs_sweep."""
    if isinstance(obj, dict):
        return types.SimpleNamespace(**{k: obj[k] for k in ("obs_sweep", "rel_recon_error", "compress_weight") if k in obj})
    return obj


def single_sweeps(core, W, H, sparsities, bits_list, groupsize=128):
    """Pruning-only sweeps (one per sparsity) and quantization-only sweeps (one per bit width).

    Returns P = {s: (W_P, M_P)} and Q = {b: (W_Q, G_Q)}; every arm re-uses these.
    """
    core = as_core(core)
    P, Q = {}, {}
    for s in sparsities:
        Wp, mP, _ = core.obs_sweep(W, H, sparsity=s, bits=16, groupsize=groupsize)
        P[s] = (Wp, mP)
    for b in bits_list:
        Wq, _, gQ = core.obs_sweep(W, H, sparsity=0.0, bits=b, groupsize=groupsize)
        Q[b] = (Wq, gQ)
    return P, Q


def run_arm(core, arm, W, H, s, b, P, Q, groupsize=128):
    """Return (W_hat, mask) for one arm at sparsity s and bit width b."""
    core = as_core(core)
    Wp, mP = P[s]
    Wq, gQ = Q[b]

    def sweep(W_in, **kw):
        return core.obs_sweep(W_in, H, groupsize=groupsize, **kw)

    if arm == "joint":
        Wh, m, _ = sweep(W, sparsity=s, bits=b)
    elif arm == "p_then_q":
        Wh, m, _ = sweep(Wp, bits=b, fixed_mask=mP)
    elif arm == "q_then_p":
        Wh, m, _ = sweep(Wq, sparsity=s, bits=b, fixed_grid=gQ)
    elif arm == "q_then_p_refit":
        Wh, m, _ = sweep(Wq, sparsity=s, bits=b)
    elif arm == "q_then_p_pmask":
        Wh, m, _ = sweep(Wq, bits=b, fixed_mask=mP, fixed_grid=gQ)
    elif arm == "q_then_p_pmask_refit":
        Wh, m, _ = sweep(Wq, bits=b, fixed_mask=mP)
    elif arm == "p_then_q_stale":
        Wh, m, _ = sweep(Wp, bits=b, fixed_mask=mP, fixed_grid=gQ)
    else:
        raise ValueError(f"unknown arm {arm!r}")
    return Wh, m


@torch.no_grad()
def layer_ablation(core, W, H, cells, arms=ALL_ARMS, groupsize=128, mask_quality=True):
    """Every arm for one linear layer and several (sparsity, bits) cells, all on one shared H.

    Returns a JSON-ready dict:
      E_P[s], E_Q[b]                     normalized layer errors of the single methods
      zeros_Q[b]                         share of weights that quantization alone sets exactly to zero
      cells["s/b"][arm]                  normalized layer error of each arm
      cells["s/b"]["mask_quality_q_then_p"]  pruning-only error with the mask Q->P selects, divided by
                                         E_P with SparseGPT's own mask (1 = as good, >1 = worse)
      cells["s/b"]["mask_jaccard_q_then_p_vs_P"]  overlap of the Q->P mask with SparseGPT's mask
    """
    core = as_core(core)
    W = W.float()
    sparsities = sorted({s for s, _ in cells})
    bits_list = sorted({b for _, b in cells})
    P, Q = single_sweeps(core, W, H, sparsities, bits_list, groupsize)
    E = lambda Wh: core.rel_recon_error(W, Wh, H)
    out = {"E_P": {key_s(s): E(P[s][0]) for s in sparsities},
           "E_Q": {str(b): E(Q[b][0]) for b in bits_list},
           "zeros_Q": {str(b): (Q[b][0] == 0).float().mean().item() for b in bits_list},
           "cells": {}}
    for s, b in cells:
        c, mQP = {}, None
        for arm in arms:
            Wh, m = run_arm(core, arm, W, H, s, b, P, Q, groupsize)
            c[arm] = E(Wh)
            if arm == "q_then_p":
                mQP = m
                c["zeros_q_then_p"] = (Wh == 0).float().mean().item()
        if mask_quality and mQP is not None:
            mP = P[s][1]
            Wm, _, _ = core.obs_sweep(W, H, sparsity=0.0, bits=16, fixed_mask=mQP, groupsize=groupsize)
            c["mask_quality_q_then_p"] = E(Wm) / max(out["E_P"][key_s(s)], 1e-30)
            inter, union = (mQP & mP).sum().item(), (mQP | mP).sum().item()
            c["mask_jaccard_q_then_p_vs_P"] = inter / max(union, 1)
        out["cells"][key_cell(s, b)] = c
    return out


def key_s(s):
    return f"{s:.4f}"


def key_cell(s, b):
    return f"{s:.4f}/{b}"


@torch.no_grad()
def check_against_core(core, W, H, s, b, groupsize=128):
    """Max |difference| between run_arm and the paper's compress_weight for the three paper orders."""
    core = as_core(core)
    P, Q = single_sweeps(core, W, H, [s], [b], groupsize)
    diffs = {}
    for arm in PAPER_ARMS:
        mine, _ = run_arm(core, arm, W, H, s, b, P, Q, groupsize)
        ref, _ = core.compress_weight(W, H, s, b, arm, groupsize)
        diffs[arm] = (mine.float() - ref.float()).abs().max().item()
    return diffs


In [ ]:
%%writefile ablation_same_hessian.py
"""Order and mechanism ablation on ONE shared Hessian per layer (review of the ESANN 2027 draft).

GPU run (a free Colab T4 is enough; see ablation_colab.ipynb):
    git clone https://github.com/antounyaacoub/esann2027-compose
    python ablation_same_hessian.py --repo esann2027-compose --model HuggingFaceTB/SmolLM2-135M
CPU smoke test (no download; a tiny random Llama model):
    python ablation_same_hessian.py --repo esann2027-compose --smoke-test

What it measures
  For every linear layer of the selected decoder blocks, H = 2XX^T is computed from the DENSE model's
  activations on the paper's calibration windows (same C4 shard, same sampler, same seed). That layer's
  dense weights are then compressed with every arm of arms.py using that one H, and the paper's
  normalized layer error E = tr(D H D^T) / tr(W H W^T) is recorded. Nothing is propagated between
  layers, so E_P, E_Q and every combination of a layer share the same H. In the paper's Fig. 1 they come
  from three separate sequential runs, whose deeper layers are calibrated on different inputs; only
  block 0 sees identical inputs in all runs (the smoke test checks that this script reproduces the
  paper's own block-0 errors).

Output
  One JSON line per (model, block, linear) in --out (resume-safe: finished layers are skipped), then a
  summary per model and cell:
    interaction    I_E / (E_P + E_Q) per arm, from sums over layers as in the paper's Fig. 1, and the
                   share of layers that are sub-additive;
    mechanism      share of the Q->P penalty (sum of E_QP - E_PQ) that each ablation arm removes;
    mask quality   pruning-only error with the mask Q->P selects, relative to SparseGPT's mask (median).

Cost on a T4 (estimated from the paper's logged sweep times, about 0.7 / 1.4 / 1.9 s per layer sweep
for the 135M / 360M / 0.5B models): the defaults (3 cells, every 4th block) need 29 sweeps per layer,
roughly 20 / 40 / 40 minutes per model. --blocks all multiplies this by about four.
"""
import argparse
import importlib.util
import json
import math
import os
import random
import statistics as st
import sys
import time
from collections import defaultdict
from fractions import Fraction

import torch
import torch.nn as nn

HERE = os.path.dirname(os.path.abspath(__file__))
if HERE not in sys.path:
    sys.path.insert(0, HERE)
import arms as A  # noqa: E402


# ----------------------------------------------------------------------------- setup helpers
def load_core(repo):
    """Import the paper's src/core.py from a clone of the repository."""
    path = os.path.join(repo, "src", "core.py")
    spec = importlib.util.spec_from_file_location("esann_core", path)
    core = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(core)
    return core


def parse_cells(text):
    """'0.25:4,0.5:4,1/3:3' -> [(0.25, 4), (0.5, 4), (0.333..., 3)] (fractions kept exact, as in the paper)."""
    cells = []
    for item in text.split(","):
        s, b = item.split(":")
        cells.append((float(Fraction(s.strip())), int(b)))
    return cells


def parse_blocks(text, n_layers):
    if text == "all":
        return list(range(n_layers))
    if text.startswith("stride"):
        return list(range(0, n_layers, int(text[len("stride"):])))
    return [int(x) for x in text.split(",")]


def get_calib(tok, c4, n, seqlen, seed):
    """Identical to get_calib in the paper's notebook (cell 4): same windows for the same seed."""
    rng = random.Random(seed)
    out = []
    while len(out) < n:
        doc = c4[rng.randint(0, len(c4) - 1)]["text"]
        ids = tok(doc, return_tensors="pt").input_ids
        if ids.shape[1] <= seqlen:
            continue
        s = rng.randint(0, ids.shape[1] - seqlen - 1)
        out.append(ids[:, s:s + seqlen])
    return out


@torch.no_grad()
def dense_hessians(core, model, calib, dev, blocks):
    """H = 2XX^T for every linear of the selected blocks, from the dense model's activations.

    Same running average as the paper's compress_model, but nothing is compressed, so every arm of a
    layer sees the same H. For block 0 this is exactly the paper's Hessian.
    """
    layers = core.get_layers(model)
    Hs, cnt, hooks = {}, {}, []

    def make_hook(key):
        def hook(mod, inp, out):
            x = inp[0].detach().reshape(-1, inp[0].shape[-1]).float()
            t = x.shape[0]
            if key not in Hs:
                Hs[key] = torch.zeros(x.shape[1], x.shape[1], device=x.device)
                cnt[key] = 0
            Hs[key] *= cnt[key] / (cnt[key] + t)
            cnt[key] += t
            x = math.sqrt(2 / cnt[key]) * x
            Hs[key] += x.T @ x
        return hook

    for li in blocks:
        for name, m in layers[li].named_modules():
            if isinstance(m, nn.Linear):
                hooks.append(m.register_forward_hook(make_hook((li, name))))
    for ids in calib:
        model(ids.to(dev), use_cache=False)
    for h in hooks:
        h.remove()
    bad = {k: int((~torch.isfinite(v)).sum().item()) for k, v in Hs.items()}
    bad = {k: v for k, v in bad.items() if v}
    if bad:
        raise FloatingPointError(f"non-finite Hessian entries in {len(bad)} layers, e.g. {next(iter(bad))}: "
                                 "collect Hessians in float32 (--dtype float32)")
    return Hs


def run_metadata(repo, dtype, dev):
    import platform
    import subprocess
    md = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
          "dtype": str(dtype), "device": dev, "colab_release_tag": os.environ.get("COLAB_RELEASE_TAG")}
    try:
        import transformers
        md["transformers"] = transformers.__version__
    except Exception:
        pass
    if dev == "cuda" and torch.cuda.is_available():
        md["gpu"] = torch.cuda.get_device_name(0)
        try:
            md["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,uuid",
                                               "--format=csv,noheader"], capture_output=True, text=True,
                                              timeout=20).stdout.strip()
        except Exception:
            pass
    try:
        md["repo_commit"] = subprocess.run(["git", "-C", repo, "rev-parse", "HEAD"], capture_output=True,
                                           text=True, timeout=20).stdout.strip()
    except Exception:
        pass
    return md


# ----------------------------------------------------------------------------- summary
def summarize(rows, cells, arms, log=print):
    by_model = defaultdict(list)
    for r in rows:
        by_model[r["model"]].append(r)
    for model, R in by_model.items():
        log(f"\n=== {model}: {len(R)} linear layers from blocks {sorted({r['block'] for r in R})}")
        for s, b in cells:
            ck, sk, bk = A.key_cell(s, b), A.key_s(s), str(b)
            R2 = [r for r in R if ck in r["cells"]]
            if not R2:
                continue
            sP = sum(r["E_P"][sk] for r in R2)
            sQ = sum(r["E_Q"][bk] for r in R2)
            log(f"  {round(s * 100):d}% + {b}-bit  (E_P sum {sP:.4g}, E_Q sum {sQ:.4g})")
            for arm in arms:
                sC = sum(r["cells"][ck][arm] for r in R2)
                sub = sum(r["cells"][ck][arm] < r["E_P"][sk] + r["E_Q"][bk] for r in R2)
                log(f"    {arm:22s} I_E/(E_P+E_Q) {100 * (sC - sP - sQ) / (sP + sQ):+7.1f}%   "
                    f"sub-additive layers {sub}/{len(R2)}")
            if "q_then_p" in arms and "p_then_q" in arms:
                pen = sum(r["cells"][ck]["q_then_p"] - r["cells"][ck]["p_then_q"] for r in R2)
                log(f"    Q->P penalty over P->Q (sum of layer errors): {pen:+.4g}"
                    f" ({100 * pen / sum(r['cells'][ck]['p_then_q'] for r in R2):+.1f}% of P->Q)")
                if pen > 0:
                    for arm in A.ABLATION_ARMS:
                        if arm in arms and arm != "p_then_q_stale":
                            closed = sum(r["cells"][ck]["q_then_p"] - r["cells"][ck][arm] for r in R2)
                            log(f"      share of the penalty removed by {arm:22s} {100 * closed / pen:+6.1f}%")
                if "p_then_q_stale" in arms:
                    hurt = sum(r["cells"][ck]["p_then_q_stale"] - r["cells"][ck]["p_then_q"] for r in R2)
                    log(f"      stale grid imposed on P->Q changes its error by "
                        f"{100 * hurt / sum(r['cells'][ck]['p_then_q'] for r in R2):+.1f}%")
            mq = [r["cells"][ck].get("mask_quality_q_then_p") for r in R2]
            mq = [x for x in mq if x is not None]
            if mq:
                jac = st.median(r["cells"][ck]["mask_jaccard_q_then_p_vs_P"] for r in R2)
                log(f"    Q->P mask used for pruning alone: median error ratio {st.median(mq):.2f}x SparseGPT's mask;"
                    f" median overlap (Jaccard) {jac:.2f}")


# ----------------------------------------------------------------------------- smoke test
def smoke_test(core, args):
    """Tiny random Llama model on CPU: checks the arms against the paper's compress_weight, checks that
    block-0 errors match the paper's own compress_model, and runs the summary end to end."""
    import copy
    from transformers import LlamaConfig, LlamaForCausalLM
    torch.manual_seed(0)
    cfg = LlamaConfig(vocab_size=512, hidden_size=128, intermediate_size=320, num_hidden_layers=3,
                      num_attention_heads=4, num_key_value_heads=2, max_position_embeddings=256)
    model = LlamaForCausalLM(cfg).float().eval()
    model.config.use_cache = False
    g = torch.Generator().manual_seed(1)
    calib = [torch.randint(0, cfg.vocab_size, (1, 128), generator=g) for _ in range(8)]
    cells = [(0.25, 4), (0.5, 4), (1 / 3, 3)]
    dev = "cpu"

    # 1. arms reproduce the paper's three orders exactly
    Hs = dense_hessians(core, model, calib, dev, blocks=[0, 1, 2])
    layer0 = core.get_layers(model)[0]
    worst = 0.0
    for name, m in layer0.named_modules():
        if isinstance(m, nn.Linear):
            for s, b in cells:
                d = A.check_against_core(core, m.weight.data.float(), Hs[(0, name)], s, b)
                worst = max(worst, max(d.values()))
    print(f"[1] arms vs paper's compress_weight (block 0, 3 cells, 3 orders): max |diff| = {worst:.3e}")
    assert worst == 0.0, "arms.py does not reproduce the paper's orders"

    # 2. block-0 errors equal the paper's own sequential pipeline (block 0 sees identical inputs)
    worst_rel = 0.0
    for order, s, b in [("prune", 0.5, 16), ("quant", 0.0, 4), ("p_then_q", 0.5, 4), ("q_then_p", 0.5, 4)]:
        mc = copy.deepcopy(model)
        diags = core.compress_model(mc, calib, s, b, order, dev, verbose=False)
        for d in diags:
            if d["layer"] != 0:
                continue
            W = layer0.get_submodule(d["name"]).weight.data.float()
            H = Hs[(0, d["name"])]
            if order in ("prune", "quant"):
                P, Q = A.single_sweeps(core, W, H, [s] if order == "prune" else [], [b] if order == "quant" else [])
                Wh = P[s][0] if order == "prune" else Q[b][0]
            else:
                P, Q = A.single_sweeps(core, W, H, [s], [b])
                Wh, _ = A.run_arm(core, order, W, H, s, b, P, Q)
            mine = core.rel_recon_error(W, Wh, H)
            worst_rel = max(worst_rel, abs(mine - d["rel_err"]) / max(d["rel_err"], 1e-30))
    print(f"[2] block-0 errors vs the paper's compress_model diags: max relative diff = {worst_rel:.2e}")
    assert worst_rel < 1e-4, "block-0 Hessians do not match the paper's pipeline"

    # 3. full ablation on all blocks + summary
    rows = []
    t0 = time.time()
    for li in range(cfg.num_hidden_layers):
        for name, m in core.get_layers(model)[li].named_modules():
            if isinstance(m, nn.Linear):
                res = A.layer_ablation(core, m.weight.data.float(), Hs[(li, name)], cells)
                rows.append(dict(model="tiny-random-llama", block=li, name=name, shape=list(m.weight.shape), **res))
    print(f"[3] ablation on {len(rows)} layers x {len(cells)} cells x {len(A.ALL_ARMS)} arms in {time.time() - t0:.1f}s")
    for r in rows:
        for c in r["cells"].values():
            for arm in A.ALL_ARMS:
                assert math.isfinite(c[arm]) and c[arm] >= 0
    summarize(rows, cells, A.ALL_ARMS)
    print("\nSMOKE TEST PASSED (random weights: the numbers above test the code path, not the paper's claims)")


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--repo", required=True, help="path to a clone of esann2027-compose (for src/core.py)")
    ap.add_argument("--model", default="HuggingFaceTB/SmolLM2-135M")
    ap.add_argument("--cells", default="1/4:4,1/2:4,1/3:3", help="sparsity:bits pairs")
    ap.add_argument("--blocks", default="stride4", help="'all', 'strideK' or a list such as 0,4,8")
    ap.add_argument("--arms", default=",".join(A.ALL_ARMS))
    ap.add_argument("--n-calib", type=int, default=128)
    ap.add_argument("--seqlen", type=int, default=2048)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--dtype", default="float32", choices=["float32", "float16", "bfloat16"],
                    help="model dtype for the calibration pass (float32 avoids the fp16 overflow)")
    ap.add_argument("--out", default="ablation_same_hessian.jsonl")
    ap.add_argument("--summary-only", action="store_true")
    ap.add_argument("--smoke-test", action="store_true")
    args = ap.parse_args()

    core = load_core(args.repo)
    if args.smoke_test:
        smoke_test(core, args)
        return

    cells = parse_cells(args.cells)
    arms = tuple(a for a in args.arms.split(",") if a)
    rows = [json.loads(l) for l in open(args.out)] if os.path.exists(args.out) else []
    if args.summary_only:
        summarize([r for r in rows if r["model"] == args.model] or rows, cells, arms)
        return

    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = getattr(torch, args.dtype)
    tok = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForCausalLM.from_pretrained(args.model, dtype=dtype).to(dev).eval()
    model.config.use_cache = False
    c4 = load_dataset("allenai/c4", data_files={"validation": "en/c4-validation.00000-of-00008.json.gz"},
                      split="validation")
    calib = get_calib(tok, c4, args.n_calib, args.seqlen, args.seed)
    blocks = parse_blocks(args.blocks, len(core.get_layers(model)))
    meta = run_metadata(args.repo, dtype, dev)
    meta.update(n_calib=args.n_calib, seqlen=args.seqlen, seed=args.seed, cells=args.cells, arms=list(arms))
    print(f"{args.model}: blocks {blocks}, cells {cells}, {len(arms)} arms | {meta.get('gpu', dev)}")

    done = {(r["model"], r["block"], r["name"]) for r in rows}
    t0 = time.time()
    Hs = dense_hessians(core, model, calib, dev, blocks)
    print(f"dense Hessians for {len(Hs)} layers in {time.time() - t0:.0f}s")
    layers = core.get_layers(model)
    with open(args.out, "a") as f:
        for li in blocks:
            for name, m in layers[li].named_modules():
                if not isinstance(m, nn.Linear) or (args.model, li, name) in done:
                    continue
                t1 = time.time()
                res = A.layer_ablation(core, m.weight.data.float(), Hs[(li, name)], cells, arms)
                row = dict(model=args.model, block=li, name=name, shape=list(m.weight.shape),
                           seconds=round(time.time() - t1, 1), **res, meta=meta)
                f.write(json.dumps(row) + "\n")
                f.flush()
                rows.append(row)
                c0 = res["cells"][A.key_cell(*cells[0])]
                print(f"  block {li:2d} {name:22s} {row['seconds']:6.1f}s  " + "  ".join(
                    f"{a}={c0[a]:.3e}" for a in ("p_then_q", "q_then_p", "q_then_p_refit") if a in c0))
            for key in [k for k in Hs if k[0] == li]:
                del Hs[key]  # this block is done: free its Hessians
            if dev == "cuda":
                torch.cuda.empty_cache()
    summarize([r for r in rows if r["model"] == args.model], cells, arms)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile extra_orders.py
"""Add-ons for the paper's notebook (esann2027_compose.ipynb) or for src/core.py.

1. install_extra_orders: lets compress_model run the ablation arms of arms.py (q_then_p_refit,
   q_then_p_pmask, q_then_p_pmask_refit, p_then_q_stale) as full sequential runs, so the mechanism
   can be checked on perplexity and tasks, not only on layer error.
2. make_safeguard_strict: turns the silent non-finite-Hessian safeguard of obs_sweep into an error.
3. kl_to_dense: mean token-level KL(dense || compressed) and top-1 agreement on WikiText-2 windows.
4. lmeval_items, flips, mcnemar: per-item correctness from lm-eval (log_samples=True), the share of
   answers that change against the dense model (Dutta et al., 2024), and an exact paired test.
5. bits_per_weight: storage per weight with the sparsity mask counted (bitmap or entropy bound) and
   the per-group scale/zero overhead.
6. run_metadata: GPU, driver, GPU UUID, CUDA, package versions, Colab runtime tag, NVML
   virtualization mode (passthrough vs vGPU), compute dtype and repository commit.

Notebook use, after cell 3 (core) and cell 4 (data, load_model):
    %run -i extra_orders.py
    install_extra_orders(globals())          # compress_model now accepts the new orders
    make_safeguard_strict(globals())         # optional: fail loudly instead of degrading silently
    GRID = [("q_then_p_refit", 0.25, 3, 0), ("q_then_p_pmask", 0.25, 3, 0), ...]
Module use:
    import core; install_extra_orders(core.__dict__)
"""
import importlib.metadata
import math
import os
import subprocess
import sys

import torch

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
if HERE not in sys.path:
    sys.path.insert(0, HERE)
import arms as A  # noqa: E402

EXTRA_ORDERS = A.ABLATION_ARMS


# ----------------------------------------------------------------------------- 1-2. orders and strict mode
def install_extra_orders(ns):
    """Re-bind compress_weight in a namespace (notebook globals or core.__dict__).

    compress_model looks compress_weight up in that namespace at call time, so after this call it
    accepts the ablation orders; the paper's own orders are passed through unchanged.
    """
    original = ns["compress_weight"]
    if getattr(original, "_extra_orders", False):
        return
    core_view = lambda: A.as_core(ns)  # looked up at every call, so a later make_safeguard_strict applies too

    def compress_weight(W, H, sparsity, bits, order, groupsize=128):
        if order not in EXTRA_ORDERS:
            return original(W, H, sparsity, bits, order, groupsize)
        P, Q = A.single_sweeps(core_view(), W, H, [sparsity], [bits], groupsize)
        return A.run_arm(core_view(), order, W, H, sparsity, bits, P, Q, groupsize)

    compress_weight._extra_orders = True
    ns["compress_weight"] = compress_weight


def make_safeguard_strict(ns, counter=None):
    """Raise instead of zeroing non-finite Hessian entries (the paper's safeguard only prints a warning,
    and the notebook outputs are not kept, so a degraded run looks like a normal one)."""
    original = ns["obs_sweep"]
    if getattr(original, "_strict", False):
        return

    def obs_sweep(W, H, *args, **kwargs):
        bad = int((~torch.isfinite(H)).sum().item())
        if bad:
            if counter is not None:
                counter.append(bad)
            raise FloatingPointError(f"{bad} non-finite Hessian entries: re-run this configuration in float32")
        return original(W, H, *args, **kwargs)

    obs_sweep._strict = True
    ns["obs_sweep"] = obs_sweep


# ----------------------------------------------------------------------------- 3. KL to the dense model
@torch.no_grad()
def kl_to_dense(model, dense_model, enc_ids, seqlen, dev, max_windows=None, chunk=512):
    """Mean KL(p_dense || p_compressed) per token (nats) and top-1 agreement, on the same windows as the
    paper's perplexity. Keep a dense copy for dense_model (load_model(name)[0] in the notebook)."""
    n = enc_ids.numel() // seqlen
    if max_windows:
        n = min(n, max_windows)
    kl_sum, agree, tokens = 0.0, 0, 0
    for i in range(n):
        ids = enc_ids[:, i * seqlen:(i + 1) * seqlen].to(dev)
        lc = model(ids, use_cache=False).logits[0, :-1]
        ld = dense_model(ids, use_cache=False).logits[0, :-1]
        for a in range(0, lc.shape[0], chunk):
            pc = torch.log_softmax(lc[a:a + chunk].float(), -1)
            pd = torch.log_softmax(ld[a:a + chunk].float(), -1)
            kl_sum += (pd.exp() * (pd - pc)).sum().item()
            agree += (pd.argmax(-1) == pc.argmax(-1)).sum().item()
            tokens += pc.shape[0]
    return {"kl_nats_per_token": kl_sum / tokens, "top1_agreement": agree / tokens, "kl_tokens": tokens}


# ----------------------------------------------------------------------------- 4. per-item results, flips, McNemar
def lmeval_items(lm, tasks, limit, main_metric, num_fewshot=0):
    """Run lm-eval with per-item logging. Returns (aggregate results, {task: {doc_id: 0/1}}).

    main_metric is the notebook's MAIN_METRIC, e.g. {"hellaswag": "acc_norm,none", ...}.
    Save the per-item dict with each result line so flips and paired tests can be computed later.
    """
    from lm_eval import simple_evaluate
    out = simple_evaluate(model=lm, tasks=tasks, num_fewshot=num_fewshot, limit=limit, bootstrap_iters=0,
                          log_samples=True, verbosity="ERROR")
    items = {}
    for t in tasks:
        key = main_metric[t].split(",")[0]
        items[t] = {int(s["doc_id"]): int(round(float(s[key]))) for s in out["samples"][t]}
    return out, items


def flips(items_a, items_b):
    """Share of items whose correctness differs between two runs, per task and pooled."""
    res, n_all, d_all = {}, 0, 0
    for t in items_a:
        common = sorted(set(items_a[t]) & set(items_b.get(t, {})))
        d = sum(items_a[t][k] != items_b[t][k] for k in common)
        res[t] = d / len(common) if common else float("nan")
        n_all += len(common)
        d_all += d
    res["pooled"] = d_all / n_all if n_all else float("nan")
    return res


def mcnemar(items_a, items_b):
    """Exact two-sided McNemar test (binomial on discordant items), per task and pooled.

    Returns {task: (right in A only, right in B only, p-value)}.
    """
    def p_exact(b, c):
        n = b + c
        if n == 0:
            return 1.0
        tail = sum(math.comb(n, i) for i in range(0, min(b, c) + 1)) / 2 ** n
        return min(1.0, 2 * tail)

    res, B, C = {}, 0, 0
    for t in items_a:
        common = set(items_a[t]) & set(items_b.get(t, {}))
        b = sum(items_a[t][k] == 1 and items_b[t][k] == 0 for k in common)
        c = sum(items_a[t][k] == 0 and items_b[t][k] == 1 for k in common)
        res[t] = (b, c, p_exact(b, c))
        B, C = B + b, C + c
    res["pooled"] = (B, C, p_exact(B, C))
    return res


# ----------------------------------------------------------------------------- 5. storage
def bits_per_weight(sparsity, bits, mask="bitmap", groupsize=128, scale_bits=16, zero_bits=None):
    """Storage per original weight, in bits.

    mask='none' reproduces the paper's TCr accounting (16 * (1 - TCr)); 'bitmap' adds 1 bit per weight
    (SparseGPT's own accounting); 'entropy' adds H(s), the lower bound for an i.i.d. mask. Quantized
    formats also pay one scale (scale_bits) and one zero point (zero_bits, default = bits) per group;
    that overhead is the same for every configuration at a given b, so it does not change rankings.
    """
    s = float(sparsity)
    if bits >= 16:
        value, overhead = 16 * (1 - s), 0.0
    else:
        value = bits * (1 - s)
        overhead = (scale_bits + (bits if zero_bits is None else zero_bits)) / groupsize
    if s <= 0 or mask == "none":
        m = 0.0
    elif mask == "bitmap":
        m = 1.0
    elif mask == "entropy":
        m = 0.0 if s >= 1 else -(s * math.log2(s) + (1 - s) * math.log2(1 - s))
    else:
        raise ValueError(mask)
    return value + m + overhead


# ----------------------------------------------------------------------------- 6. metadata
def run_metadata(repo=None, dtype=None):
    """Everything needed to tell runs apart later; store it in every result line."""
    import platform
    md = {"python": platform.python_version(), "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
          "colab_release_tag": os.environ.get("COLAB_RELEASE_TAG"), "compute_dtype": str(dtype)}
    for dist in ("transformers", "lm_eval", "datasets", "accelerate"):
        try:
            md[dist] = importlib.metadata.version(dist)
        except Exception:
            md[dist] = None
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        md.update(gpu=p.name, capability=list(torch.cuda.get_device_capability(0)),
                  gpu_mem_gb=round(p.total_memory / 1e9, 2))
        try:
            q = "name,driver_version,uuid,pci.bus_id,memory.total,clocks.max.sm"
            md["nvidia_smi"] = subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader"],
                                              capture_output=True, text=True, timeout=20).stdout.strip()
        except Exception:
            pass
        try:
            import pynvml  # pip install nvidia-ml-py
            pynvml.nvmlInit()
            mode = pynvml.nvmlDeviceGetVirtualizationMode(pynvml.nvmlDeviceGetHandleByIndex(0))
            md["virtualization_mode"] = {0: "none (bare metal)", 1: "passthrough", 2: "vgpu",
                                         3: "host_vgpu", 4: "host_vsga"}.get(mode, str(mode))
        except Exception as e:
            md["virtualization_mode"] = f"unavailable ({type(e).__name__})"
    if repo:
        try:
            md["repo_commit"] = subprocess.run(["git", "-C", repo, "rev-parse", "HEAD"], capture_output=True,
                                               text=True, timeout=20).stdout.strip()
        except Exception:
            pass
    return md


In [ ]:
# 2. smoke test on a tiny random model (about 15 s): the arms must reproduce the paper's orders exactly
!python ablation_same_hessian.py --repo esann2027-compose --smoke-test


In [ ]:
# 3. the ablation on the three models (results on Drive)
import os
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/esann2027/ablation_same_hessian.jsonl'
os.makedirs(os.path.dirname(OUT), exist_ok=True)
CELLS, BLOCKS = '1/4:4,1/2:4,1/3:3', 'stride4'
MODELS = ['HuggingFaceTB/SmolLM2-135M', 'HuggingFaceTB/SmolLM2-360M', 'Qwen/Qwen2.5-0.5B']
for m in MODELS:
    !python ablation_same_hessian.py --repo esann2027-compose --model {m} --cells {CELLS} --blocks {BLOCKS} --out {OUT}


In [ ]:
# 4. summary per model (re-run any time)
for m in MODELS:
    !python ablation_same_hessian.py --repo esann2027-compose --model {m} --cells {CELLS} --out {OUT} --summary-only


## Full sequential runs with the ablation arms
In `esann2027_compose.ipynb`, after cells 3 and 4, copy `arms.py` and `extra_orders.py` next to it and run:
```python
%run -i extra_orders.py
install_extra_orders(globals())
make_safeguard_strict(globals())
GRID = [('q_then_p_refit', 0.25, 3, 0), ('q_then_p_pmask', 0.25, 3, 0), ('q_then_p_refit', 0.5, 4, 0), ('q_then_p_pmask', 0.5, 4, 0)]
```
and store `kl_to_dense(...)`, the per-item dict from `lmeval_items(...)` and `run_metadata(...)` with every result line.
